## Download text file

In [5]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !wget https://raw.githubusercontent.com/danielmiessler/SecLists/master/Passwords/Common-Credentials/10k-most-common.txt -O 10k-most-common.txt

## Example code

In [6]:
import hashlib

m=hashlib.sha1(b"Chulalongkorn").hexdigest()
m2=hashlib.sha1(b"Chulalongkorn University").hexdigest()
print(m)
print(m2)
m=hashlib.md5(b"Chulalongkorn").hexdigest()
m2=hashlib.md5(b"Chulalongkorn University").hexdigest()
print(m)
print(m2)

ca8a68498ae67cd14c15f5ebf043633224005759
a16c5b03cf3aca5c2f20169b4caa909d5c2f07ad
46fa3b56c660faff420190c18c98a56b
cc3fed293eb73ca7d3597a31259df950


## Implement Hash Function with Pandas

In [7]:
import pandas as pd
import time

original_value = 'd54cc1fe76f5186380a0939d2fc1723c44e8a5f7'
txt_path = '10k-most-common.txt'

df = pd.read_csv(txt_path, header=None, names=['word'])

In [8]:
pair_num_lett = [('o', '0'), ('l', '1'), ('i', '1')]

def replace_number_to_letter(word, number, letter):
  if str(number) in str(word):
    return word.replace(number, letter)
  return "-"

def get_uppercase_list(word):
  last_uppercase_word = word.upper()
  word_list = [word]
  for string_word in word_list:
    if word_list[-1] == last_uppercase_word:
      break
    for l in str(string_word):
        if l in "123456789" or l.isupper():
          continue
        new_word = string_word.replace(l, l.upper())
        word_list.append(new_word)
  return word_list

def hash_word(word):
  return hashlib.sha1(str(word).encode()).hexdigest()

start_time = time.time()

for p, n in pair_num_lett:
  p, n = pair_num_lett[0]
  new_df = pd.DataFrame()
  new_df['word'] = df['word'].apply(
    lambda x: replace_number_to_letter(x, p, n)
  ).reset_index(drop=True)
  new_df = new_df.drop_duplicates()
  df = pd.concat([df, new_df]).reset_index(drop=True)

for i in range(len(df)//1000+1):
  start_idx = i*1000
  stop_idx = (i+1)*1000

  word_list = df['word'][start_idx:stop_idx].apply(
      lambda x: get_uppercase_list(x)
    ).reset_index(drop=True)
  
  df = df.drop([start_idx+1, stop_idx])

  combine_ls = []
  for sub_ls in word_list.values.tolist():
    combine_ls += sub_ls
  
  del word_list
  
  new_df = pd.DataFrame({'password': combine_ls})
  new_df['hashed_password'] = new_df['password'].apply(hash_word)
  new_df.to_csv(f'hash_tables/hash{stop_idx}.csv', index=False)
  print(f"Save hash{stop_idx}.csv lines:{len(new_df)}")
  
  del combine_ls
  del new_df

elapsed_time = time.time() - start_time

print(f"elapsed_time: {elapsed_time}")

Save hash1000.csv lines:3969762
Save hash2000.csv lines:13152641
Save hash3000.csv lines:12226362


KeyboardInterrupt: 

In [ ]:
# def get_uppercase_list(word):
#   last_uppercase_word = word.upper()
#   word_list = [word]
#   for string_word in word_list:
#     if word_list[-1] == last_uppercase_word:
#       break
#     for l in str(string_word):
#         if l in "123456789" or l.isupper():
#           continue
#         new_word = string_word.replace(l, l.upper())
#         word_list.append(new_word)
#   return word_list

# # for i in range(10):
# save10 = df['word'][:3000].apply(
#     lambda x: get_uppercase_list(x)
#   ).reset_index(drop=True)

In [ ]:
# new_ls = []
# for sub_ls in save10.values.tolist():
#   new_ls += sub_ls

# print(len(new_ls))

29372399


In [ ]:
# print(df[df['hashed_password'] == original_value]['word'])